In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [3]:
input_size = 28        # One row contains 28 pixels
sequence_length = 28   # Total rows

hidden_size = 128
num_layers = 2

num_classes = 10

batch_size = 64

learning_rate = 0.001

epochs = 5

In [4]:
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

100%|██████████| 9.91M/9.91M [00:24<00:00, 412kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 40.0kB/s]
100%|██████████| 1.65M/1.65M [00:08<00:00, 206kB/s] 
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.98MB/s]


In [5]:
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [6]:
class LSTMModel(nn.Module):

    def __init__(self):
        super(LSTMModel, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):

        # x shape:
        # (batch_size, 1, 28, 28)

        x = x.squeeze(1)

        # x becomes
        # (batch_size, 28, 28)

        output, (hidden, cell) = self.lstm(x)

        # hidden shape:
        # (num_layers, batch_size, hidden_size)

        out = hidden[-1]

        output = self.fc(out)

        return output

In [7]:
model = LSTMModel().to(device)

print(model)

LSTMModel(
  (lstm): LSTM(28, 128, num_layers=2, batch_first=True)
  (fc): Linear(in_features=128, out_features=10, bias=True)
)


In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=learning_rate
)

In [9]:
for epoch in range(epochs):

    model.train()

    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch [{epoch+1}/{epochs}] Loss: {running_loss/len(train_loader):.4f}"
    )

Epoch [1/5] Loss: 0.4026
Epoch [2/5] Loss: 0.1093
Epoch [3/5] Loss: 0.0724
Epoch [4/5] Loss: 0.0560
Epoch [5/5] Loss: 0.0470


In [10]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item() 

In [11]:
accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 98.47%
